# Skipgram Classification Naive bayes

## Preprocessing

### Case Folding

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

file_path = r"dataset_detik_sport_finance.xlsx"
df = pd.read_excel(file_path)

df["teks_lower"] = df["isi_berita"].str.lower()

df[["isi_berita", "teks_lower"]].head()

,isi_berita,teks_lower
0,"Pegokar muda Indonesia, Morgan Holindo, tak pu...","pegokar muda indonesia, morgan holindo, tak pu..."
1,Pacuan kuda Indonesia mulai diarahkan menuju p...,pacuan kuda indonesia mulai diarahkan menuju p...
2,"Jelang Asian Games 2026, dukungan untuk 432 at...","jelang asian games 2026, dukungan untuk 432 at..."
3,Pupus sudah harapan Sean Gelael dan rekan-reka...,pupus sudah harapan sean gelael dan rekan-reka...
4,Barra Ghaisan Zeinatma menunjukkan perkembanga...,barra ghaisan zeinatma menunjukkan perkembanga...


### Cleaning

In [2]:
import string
import re

def cleaning(text):
    text = str(text)
    # pola nama author didalam kurung
    text = re.sub(
        r'\([a-z]{2,5}/[a-z]{2,5}\)',
        ' ',
        text,
        flags=re.IGNORECASE
    )
    
    # Hapus tanda baca
    text = text.translate(str.maketrans("", "", string.punctuation))
    
    # Hapus karakter selain huruf dan spasi
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    
    # Hapus spasi berlebih
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

df["teks_clean"] = df["teks_lower"].apply(cleaning)

df[["teks_lower", "teks_clean"]].head()

,teks_lower,teks_clean
0,"pegokar muda indonesia, morgan holindo, tak pu...",pegokar muda indonesia morgan holindo tak puas...
1,pacuan kuda indonesia mulai diarahkan menuju p...,pacuan kuda indonesia mulai diarahkan menuju p...
2,"jelang asian games 2026, dukungan untuk 432 at...",jelang asian games dukungan untuk atlet indone...
3,pupus sudah harapan sean gelael dan rekan-reka...,pupus sudah harapan sean gelael dan rekanrekan...
4,barra ghaisan zeinatma menunjukkan perkembanga...,barra ghaisan zeinatma menunjukkan perkembanga...


### Tokenisasi

In [3]:
df["tokens"] = df["teks_clean"].apply(lambda x: x.split())

df[["teks_clean", "tokens"]].head()

,teks_clean,tokens
0,pegokar muda indonesia morgan holindo tak puas...,"[pegokar, muda, indonesia, morgan, holindo, ta..."
1,pacuan kuda indonesia mulai diarahkan menuju p...,"[pacuan, kuda, indonesia, mulai, diarahkan, me..."
2,jelang asian games dukungan untuk atlet indone...,"[jelang, asian, games, dukungan, untuk, atlet,..."
3,pupus sudah harapan sean gelael dan rekanrekan...,"[pupus, sudah, harapan, sean, gelael, dan, rek..."
4,barra ghaisan zeinatma menunjukkan perkembanga...,"[barra, ghaisan, zeinatma, menunjukkan, perkem..."


### Stopword Removal

In [4]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stopwords]

df["tokens_no_stopword"] = df["tokens"].apply(remove_stopwords)

df[["tokens", "tokens_no_stopword"]].head()

,tokens,tokens_no_stopword
0,"[pegokar, muda, indonesia, morgan, holindo, ta...","[pegokar, muda, indonesia, morgan, holindo, ta..."
1,"[pacuan, kuda, indonesia, mulai, diarahkan, me...","[pacuan, kuda, indonesia, mulai, diarahkan, me..."
2,"[jelang, asian, games, dukungan, untuk, atlet,...","[jelang, asian, games, dukungan, atlet, indone..."
3,"[pupus, sudah, harapan, sean, gelael, dan, rek...","[pupus, harapan, sean, gelael, rekanrekannya, ..."
4,"[barra, ghaisan, zeinatma, menunjukkan, perkem...","[barra, ghaisan, zeinatma, menunjukkan, perkem..."


### Stemming

In [5]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming(tokens):
    return [stemmer.stem(word) for word in tokens]

df["tokens_stem"] = df["tokens_no_stopword"].apply(stemming)

df[["tokens_no_stopword", "tokens_stem"]].head()

df["teks_preprocessed"] = df["tokens_stem"].apply(
    lambda x: " ".join(x)
)

df[["isi_berita", "teks_preprocessed", "jenis_berita"]].head()

,isi_berita,teks_preprocessed,jenis_berita
0,"Pegokar muda Indonesia, Morgan Holindo, tak pu...",gokar muda indonesia morgan holindo tak puas c...,sport
1,Pacuan kuda Indonesia mulai diarahkan menuju p...,pacu kuda indonesia mulai arah tuju panggung d...,sport
2,"Jelang Asian Games 2026, dukungan untuk 432 at...",jelang asi games dukung atlet indonesia terus ...,sport
3,Pupus sudah harapan Sean Gelael dan rekan-reka...,pupus harap sean gelael rekanrekannya team wrt...,sport
4,Barra Ghaisan Zeinatma menunjukkan perkembanga...,barra ghaisan zeinatma tunjuk kembang positif ...,sport


## Menyiapkan 2 Skenario Data (Tanpa vs Dengan Stopword Removal)

Dua skenario yang akan dibandingkan:

1. **Tanpa stopword removal** &rarr; menggunakan `tokens` (hasil tokenisasi) yang langsung di-*stem*.
2. **Dengan stopword removal** &rarr; menggunakan `tokens_no_stopword` yang sudah di-*stem* (`tokens_stem` / `teks_preprocessed`, sudah dibuat di bagian Stemming di atas).

In [6]:
# Skenario 1: TANPA stopword removal -> stemming langsung dari token hasil tokenisasi
df["tokens_stem_full"] = df["tokens"].apply(stemming)
df["teks_preprocessed_full"] = df["tokens_stem_full"].apply(lambda x: " ".join(x))

# Skenario 2: DENGAN stopword removal -> sudah tersedia di df["tokens_stem"] / df["teks_preprocessed"]

skenario = {
    "tanpa_stopword_removal": df["tokens_stem_full"],
    "dengan_stopword_removal": df["tokens_stem"],
}

df[["tokens_stem_full", "tokens_stem"]].head()


,tokens_stem_full,tokens_stem
0,"[gokar, muda, indonesia, morgan, holindo, tak,...","[gokar, muda, indonesia, morgan, holindo, tak,..."
1,"[pacu, kuda, indonesia, mulai, arah, tuju, pan...","[pacu, kuda, indonesia, mulai, arah, tuju, pan..."
2,"[jelang, asi, games, dukung, untuk, atlet, ind...","[jelang, asi, games, dukung, atlet, indonesia,..."
3,"[pupus, sudah, harap, sean, gelael, dan, rekan...","[pupus, harap, sean, gelael, rekanrekannya, te..."
4,"[barra, ghaisan, zeinatma, tunjuk, kembang, po...","[barra, ghaisan, zeinatma, tunjuk, kembang, po..."


### Word Embedding Skipgram

In [8]:
%pip install gensim

  Using cached smart_open-8.0.1-py3-none-any.whl.metadata (24 kB)
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   -- ------------------------------------- 1.6/24.4 MB 11.9 MB/s eta 0:00:02
   ------ --------------------------------- 3.9/24.4 MB 11.7 MB/s eta 0:00:02
   --------- ------------------------------ 6.0/24.4 MB 11.2 MB/s eta 0:00:02
   ------------- -------------------------- 8.1/24.4 MB 10.7 MB/s eta 0:00:02
   ---------------- ----------------------- 10.2/24.4 MB 10.4 MB/s eta 0:00:02
   -------------------- ------------------- 12.3/24.4 MB 10.4 MB/s eta 0:00:02
   ------------------------ --------------- 14.7/24.4 MB 10.5 MB/s eta 0:00:01
   --------------------------- ------------ 17.0/24.4 MB 10.6 MB/s eta 0:00:01
   -------------------------------- ------- 19.7/24.4 MB 10.7 MB/s eta 0:00:01
   ------------------------------------ --- 22.0/24.4 MB 10.8 MB/s eta 0:00:01
   ---------------------------------------  24.4/24.4 MB 10.9 MB/s eta 0:00:01


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from gensim.models import Word2Vec
def train_skipgram(corpus_tokens, vector_size=100, window=5, min_count=1, epochs=100, seed=42):
    """Melatih model Word2Vec dengan arsitektur skip-gram (sg=1)."""
    model = Word2Vec(
        sentences=corpus_tokens.tolist(),
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        sg=1,          # 1 = skip-gram, 0 = CBOW
        epochs=epochs,
        seed=seed,
        workers=1,
    )
    return model

# Latih satu model skip-gram untuk masing-masing skenario (parameter dasar)
skipgram_models = {
    nama: train_skipgram(tokens_col, vector_size=100, window=5)
    for nama, tokens_col in skenario.items()
}

for nama, model in skipgram_models.items():
    print(f"{nama}: ukuran vocabulary = {len(model.wv):,}")


tanpa_stopword_removal: ukuran vocabulary = 5,346
dengan_stopword_removal: ukuran vocabulary = 5,299


In [10]:
contoh_model = skipgram_models["dengan_stopword_removal"]
contoh_kata = contoh_model.wv.index_to_key[0]

print("Contoh kata:", contoh_kata)
print("Vektor (5 dimensi pertama):", contoh_model.wv[contoh_kata][:5])
print("Kata paling mirip konteksnya:")
for kata, skor in contoh_model.wv.most_similar(contoh_kata, topn=5):
    print(f"  {kata:<20s} {skor:.4f}")


Contoh kata: jadi
Vektor (5 dimensi pertama): [-0.17439373  0.13864145  0.1338177   0.00406454  0.01997869]
Kata paling mirip konteksnya:
  akurasi              0.5442
  sebut                0.5059
  dagestan             0.4913
  ready                0.4828
  penting              0.4743


## Representasi Dokumen (Rata-rata Vektor Kata)

In [11]:
import numpy as np

def document_vector(model, tokens):
    """Rata-rata vektor kata dalam satu dokumen. Kata di luar vocab diabaikan."""
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

def build_feature_matrix(model, tokens_series):
    return np.vstack([document_vector(model, toks) for toks in tokens_series])

X_no_sw   = build_feature_matrix(skipgram_models["tanpa_stopword_removal"], skenario["tanpa_stopword_removal"])
X_with_sw = build_feature_matrix(skipgram_models["dengan_stopword_removal"], skenario["dengan_stopword_removal"])

y = df["jenis_berita"]

print("Shape fitur (tanpa stopword removal) :", X_no_sw.shape)
print("Shape fitur (dengan stopword removal):", X_with_sw.shape)


Shape fitur (tanpa stopword removal) : (200, 100)
Shape fitur (dengan stopword removal): (200, 100)


## Klasifikasi Naive Bayes

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

def evaluate_naive_bayes(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    clf = GaussianNB()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, zero_division=0)
    return acc, report, clf

hasil = {}
for nama, X in [("tanpa_stopword_removal", X_no_sw), ("dengan_stopword_removal", X_with_sw)]:
    acc, report, clf = evaluate_naive_bayes(X, y)
    hasil[nama] = {"akurasi": acc, "report": report, "model": clf}
    print(f"=== Skenario: {nama} ===")
    print(f"Akurasi: {acc:.4f}")
    print(report)
    print()


=== Skenario: tanpa_stopword_removal ===
Akurasi: 0.9750
              precision    recall  f1-score   support

     finance       0.95      1.00      0.98        20
       sport       1.00      0.95      0.97        20

    accuracy                           0.97        40
   macro avg       0.98      0.97      0.97        40
weighted avg       0.98      0.97      0.97        40


=== Skenario: dengan_stopword_removal ===
Akurasi: 0.9750
              precision    recall  f1-score   support

     finance       0.95      1.00      0.98        20
       sport       1.00      0.95      0.97        20

    accuracy                           0.97        40
   macro avg       0.98      0.97      0.97        40
weighted avg       0.98      0.97      0.97        40




## Eksperimen: Pengaruh Window Size & Vector Size (Dimensi Embedding) terhadap Akurasi

In [14]:
window_list = [2, 5, 8]
vector_size_list = [50, 100, 200]

eksperimen_hasil = []

for nama_skenario, tokens_col in skenario.items():
    for window in window_list:
        for vector_size in vector_size_list:
            model = train_skipgram(tokens_col, vector_size=vector_size, window=window)
            X = build_feature_matrix(model, tokens_col)
            acc, _, _ = evaluate_naive_bayes(X, y)
            eksperimen_hasil.append({
                "skenario": nama_skenario,
                "window": window,
                "vector_size": vector_size,
                "akurasi": acc,
            })

df_eksperimen = pd.DataFrame(eksperimen_hasil).sort_values("akurasi", ascending=False).reset_index(drop=True)
display(df_eksperimen)


,skenario,window,vector_size,akurasi
0,dengan_stopword_removal,5,50,1.000
1,tanpa_stopword_removal,5,50,1.000
2,tanpa_stopword_removal,2,100,0.975
3,tanpa_stopword_removal,2,50,0.975
4,tanpa_stopword_removal,5,100,0.975
5,tanpa_stopword_removal,2,200,0.975
6,tanpa_stopword_removal,8,100,0.975
7,dengan_stopword_removal,2,50,0.975
8,dengan_stopword_removal,2,100,0.975
9,tanpa_stopword_removal,5,200,0.975
